# R3maJ — Headless Training + Native W&B

The executable owns W&B telemetry. The notebook only supplies the secret, builds the C++ core, restores Drive state, and starts training. Rendering is disabled in the C++ learner configuration.

In [ ]:
# 1. Clone / update repository
import os, subprocess, shutil, glob, time
ROOT='/content/R3maJ'
REPO='https://github.com/vfxjamer/R3maJ.git'
if not os.path.isdir(os.path.join(ROOT,'.git')):
    subprocess.run(['git','clone',REPO,ROOT],check=True)
else:
    subprocess.run(['git','-C',ROOT,'pull'],check=False)
print('ROOT:',ROOT)
os.chdir(ROOT)

In [ ]:
# 2. Secure W&B configuration
from google.colab import userdata
key=userdata.get('WANDB_API_KEY')
if not key:
    raise RuntimeError('Add WANDB_API_KEY to Colab Secrets.')
os.environ['WANDB_API_KEY']=key.strip()
os.environ['WANDB_MODE']='online'
print('W&B secret loaded. No Python wandb.login() is used.')

In [ ]:
# 3. Mount Drive and restore checkpoints/replays
from google.colab import drive
drive.mount('/content/drive',force_remount=False)
DRIVE_ROOT='/content/drive/MyDrive/R3maJ'
BUILD=os.path.join(ROOT,'build')
os.makedirs(os.path.join(BUILD,'checkpoints'),exist_ok=True)
drive_ckpt=os.path.join(DRIVE_ROOT,'checkpoints')
local_ckpt=os.path.join(BUILD,'checkpoints')
drive_replay=os.path.join(DRIVE_ROOT,'serialized_replays.bin')
local_replay=os.path.join(BUILD,'serialized_replays.bin')
if os.path.isdir(drive_ckpt):
    dirs=[d for d in glob.glob(os.path.join(drive_ckpt,'*')) if os.path.isdir(d)]
    dirs.sort(key=lambda d:int(os.path.basename(d)) if os.path.basename(d).isdigit() else -1)
    if dirs:
        newest=dirs[-1]; dest=os.path.join(local_ckpt,os.path.basename(newest))
        if not os.path.exists(dest): shutil.copytree(newest,dest)
if os.path.exists(drive_replay) and not os.path.exists(local_replay): shutil.copy2(drive_replay,local_replay)
print('checkpoints:',sorted(os.listdir(local_ckpt)))
print('replay:',os.path.exists(local_replay))

In [ ]:
# 4. Build the current C++ core
os.chdir(ROOT)
subprocess.run(['cmake','-S','.','-B','build','-DCMAKE_BUILD_TYPE=Release'],check=True)
subprocess.run(['cmake','--build','build','--config','Release','-j2'],check=True)
BUILD=os.path.join(ROOT,'build')
print('Build complete:',os.path.join(BUILD,'R3maJ'))

In [ ]:
# 5. Verify headless/native telemetry configuration
pm=open(os.path.join(ROOT,'src','PhaseManager.cpp'),errors='ignore').read()
main=open(os.path.join(ROOT,'src','main.cpp'),errors='ignore').read()
assert 'cfg.renderMode=false' in pm, 'Renderer is still enabled in PhaseManager.cpp'
assert 'cfg.sendMetrics' in main, 'Native W&B telemetry is missing from main.cpp'
print('OK: render mode disabled in core')
print('OK: native W&B telemetry enabled in core')

In [ ]:
# 6. Start R3maJ — LIVE training output
import torch
os.chdir(BUILD)
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
REPLAY_ARG=['--replays','serialized_replays.bin'] if os.path.exists('serialized_replays.bin') else []
CMD=['stdbuf','-oL','-eL','./R3maJ','--device',DEVICE,'--phase','-1','--save-dir','checkpoints','--games','164']+REPLAY_ARG
print('BASE CMD:',' '.join(CMD))
print('W&B metrics are emitted by src/main.cpp. No Python W&B logger is used.')
print('Training output is streamed below and also saved to train.log.')
with open('train.log','a',buffering=1) as log:
    proc=subprocess.Popen(CMD,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,start_new_session=True)
    for line in proc.stdout:
        print(line,end='')
        log.write(line)
    rc=proc.wait()
print('R3maJ exit code:',rc)

In [ ]:
# 7. Optional: tail the saved training log
import subprocess, os
os.chdir(BUILD)
subprocess.run(['tail','-n','80','train.log'])